In [1]:
!pip install tensorflow


##LSTM

In [2]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [15]:
# Sample dataset (Shakespeare text)
text = """To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer The slings and arrows of outrageous fortune, Or to take arms against a sea of troubles"""

In [16]:
# Tokenize text

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
print(total_words)

27


In [26]:
# Prepare sequences

input_sequences = []

for line in text.split("\n"):
    words = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(words)):
        input_sequences.append(words[:i+1])


In [8]:
text.split("\n")

["To be, or not to be, that is the question: Whether 'tis nobler in the mind to suffer The slings and arrows of outrageous fortune, Or to take arms against a sea of troubles"]

In [14]:
input_sequences

array([[ 0,  0,  0, ...,  0,  1,  3],
       [ 0,  0,  0, ...,  1,  3,  4],
       [ 0,  0,  0, ...,  3,  4,  6],
       ...,
       [ 0,  0,  1, ..., 23, 24, 25],
       [ 0,  1,  3, ..., 24, 25,  5],
       [ 1,  3,  4, ..., 25,  5, 26]], dtype=int32)

In [27]:
# Pad sequences
max_sequence_length = max([len(seq) for seq in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_length, padding='pre')

In [28]:
# Features & labels

X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)


In [29]:
# Build LSTM model

model = Sequential([
                    Embedding(total_words, 10, input_length=max_sequence_length-1),
                    LSTM(100),
                    Dense(total_words, activation='softmax')])

In [30]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [31]:
# Train model
model.fit(X, y, epochs=100, verbose=1)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 46ms/step - accuracy: 0.0612 - loss: 3.2955
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.0919 - loss: 3.2923
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.1225 - loss: 3.2903
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.0919 - loss: 3.2882
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.0612 - loss: 3.2865
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.0612 - loss: 3.2849
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.0612 - loss: 3.2825
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.0612 - loss: 3.2805
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.0919 - loss: 3.2787
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.0919 - loss: 3.2777
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.0919 - loss: 3.2741
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.0814 - lo

In [32]:
# Generate text
seed_text = "To be"
for _ in range(5):
    tokenized = tokenizer.texts_to_sequences([seed_text])[0]
    tokenized = pad_sequences([tokenized], maxlen=max_sequence_length-1, padding='pre')
    predicted = np.argmax(model.predict(tokenized), axis=-1)
    output_word = tokenizer.index_word.get(predicted[0], "")
    seed_text += " " + output_word

print(seed_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 204ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
To be not not not not not
